# C01_E01 — Lazo abierto vs lazo cerrado en depósito

**Capítulo:** 1 — Qué significa controlar un sistema industrial  
**Identificador:** `C01_E01`  
**Objetivo pedagógico:** Visualizar el efecto cualitativo de cerrar el lazo con realimentación negativa.  
**Librerías:** numpy, scipy.integrate, matplotlib

## Ejemplo industrial

Depósito con descarga proporcional al nivel: dh/dt = (Qin − k·h)/A. Mismo proceso bajo dos modos de operación (manual y automático) para mostrar el cambio de respuesta.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Importaciones y parámetros

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import os

np.random.seed(0)
A, k = 2.0, 0.1            # area (m^2), descarga (m^2/min)
Qin_nom = 0.5              # caudal nominal (m^3/min)
SP = 4.0                   # consigna de nivel (m)
Kc = 0.4                   # ganancia proporcional

## 2. Modelos lazo abierto y lazo cerrado

In [2]:
def lazo_abierto(t, h):
    return (Qin_nom - k*h[0]) / A

def lazo_cerrado(t, h):
    e = SP - h[0]
    Qin = max(0.0, Qin_nom + Kc*e)
    return (Qin - k*h[0]) / A

t_eval = np.linspace(0, 200, 500)
sol_oa = solve_ivp(lazo_abierto, (0, 200), [0.5], t_eval=t_eval)
sol_lc = solve_ivp(lazo_cerrado, (0, 200), [0.5], t_eval=t_eval)

## 3. Visualización

In [3]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sol_oa.t, sol_oa.y[0], label="Lazo abierto")
ax.plot(sol_lc.t, sol_lc.y[0], label="Lazo cerrado P")
ax.axhline(SP, ls="--", color="gray", label="Consigna")
ax.set_xlabel("Tiempo (min)")
ax.set_ylabel("Nivel h (m)")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Depósito: lazo abierto vs lazo cerrado P")
out = '../../figures/cap01/fig_C01_F01_lazo_abierto_vs_cerrado.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120)
plt.show()

## 4. Interpretación

El lazo abierto se estabiliza en `h_eq = Qin/k = 5 m`, no en la consigna. El lazo cerrado proporcional alcanza un nivel cercano a SP = 4 m con un error en régimen permanente (residuo del controlador P sin acción integral). Añadir acción integral lo elimina (Capítulos 5 y 9).